# Prétraitement des données

In [ ]:
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from pandas import DataFrame
from PIL import Image
from sklearn.base import BaseEstimator
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from typing import Optional, Dict, Any

In [4]:
random_state = 42
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [5]:
def train(
    model: BaseEstimator,
    X_train: DataFrame,
    y_train: DataFrame,
    X_test: DataFrame,
    y_test: DataFrame,
    param_grid: Optional[Dict[str, Any]] = None,
    run_name: Optional[str] = None,
    tags: Optional[Dict[str, str]] = None,
    cv: int = 5,
    scoring: Optional[str] = None,
    n_jobs: Optional[int] = None
) -> BaseEstimator:
    """
    Entraîne un modèle avec ou sans GridSearchCV et log automatiquement dans MLflow.

    Args:
        model: Un estimateur scikit-learn.
        X_train, y_train: Données d'entraînement.
        X_test, y_test: Données de test pour evaluation finale.
        param_grid: Dictionnaire d'hyperparamètres pour GridSearchCV (None pour entraînement direct).
        run_name: Nom du run MLflow.
        tags: Tags MLflow à ajouter au run.
        cv: Nombre de folds pour GridSearchCV.
        scoring: Métrique de scoring pour GridSearchCV.
        n_jobs: Nombre de jobs parallèles pour GridSearchCV.

    Returns:
        Le meilleur estimateur entraîné.
    """

    mlflow.set_experiment("Imo")

    with mlflow.start_run(run_name=run_name, tags=tags) as run:
        mlflow.sklearn.autolog(log_models=False, max_tuning_runs=1)
        if param_grid:
            # Recherche par grille
            grid = GridSearchCV(
                estimator=model,
                param_grid=param_grid,
                cv=cv,
                scoring=scoring,
                n_jobs=n_jobs
            )
            grid.fit(X_train, y_train)
            best_est = grid.best_estimator_
            # Log des meilleurs paramètres
            mlflow.log_params(grid.best_params_)
            mlflow.set_tag("best_params", str(grid.best_params_))
        else:
            # Entraînement direct
            model.fit(X_train, y_train)
            best_est = model

        # Évaluation finale sur le jeu de test
        test_score = best_est.score(X_test, y_test)
        mlflow.log_metric("test_score", test_score)

    return best_est


# Linear Model

In [6]:
lr = LinearRegression()
train(lr, X_train_scaled, y_train, X_test_scaled, y_test, run_name="LinearRegression")

2025/08/07 16:49:30 INFO mlflow.tracking.fluent: Experiment with name 'Imo' does not exist. Creating a new experiment.


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [7]:
lr = LinearRegression()
train(lr, X_train_scaled, y_train, X_test_scaled, y_test, run_name="LinearRegression_scaled")

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [8]:
params_grid = { "alpha": np.logspace(-3, 3, 10) }

In [9]:
model = { 'Ridge': Ridge(random_state=random_state), 'Lasso': Lasso(random_state=random_state)}
for name, mod in model.items():
    train(mod, X_train, y_train, X_test, y_test, param_grid=params_grid, run_name=name)
    train(mod, X_train_scaled, y_train, X_test_scaled, y_test, param_grid=params_grid, run_name=f"{name}_scaled")

2025/08/07 16:49:34 INFO mlflow.sklearn.utils: Logging the best run, 9 runs will be omitted.
2025/08/07 16:49:35 INFO mlflow.sklearn.utils: Logging the best run, 9 runs will be omitted.
2025/08/07 16:49:41 INFO mlflow.sklearn.utils: Logging the best run, 9 runs will be omitted.
2025/08/07 16:49:46 INFO mlflow.sklearn.utils: Logging the best run, 9 runs will be omitted.


In [10]:
params_grid = { "alpha": np.logspace(-3, 3, 10), "l1_ratio": np.linspace(0.001, 0.999, 10) }

In [11]:
train(ElasticNet(random_state=random_state), X_train, y_train, X_test, y_test, param_grid=params_grid, run_name="ElasticNet")
train(ElasticNet(random_state=random_state), X_train_scaled, y_train, X_test_scaled, y_test, param_grid=params_grid, run_name="ElasticNet_scaled")

2025/08/07 16:50:43 INFO mlflow.sklearn.utils: Logging the best run, 99 runs will be omitted.
2025/08/07 16:51:26 INFO mlflow.sklearn.utils: Logging the best run, 99 runs will be omitted.


,alpha,np.float64(0.001)
,l1_ratio,np.float64(0.999)
,fit_intercept,True
,precompute,False
,max_iter,1000
,copy_X,True
,tol,0.0001
,warm_start,False
,positive,False
,random_state,42
,selection,'cyclic'


# tree

In [12]:
params_grid = { "max_depth": [None, 3, 5],
                "n_estimators": [100, 150] }

In [13]:
train(RandomForestRegressor(random_state=random_state), X_train, y_train, X_test, y_test, param_grid=params_grid, run_name="RandomForest")

2025/08/07 16:58:09 INFO mlflow.sklearn.utils: Logging the best run, 5 runs will be omitted.


,n_estimators,150
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [14]:
params_grid = { "max_depth": [None, 3, 5],
                "n_estimators": [100, 150],
                 "learning_rate": [0.1, 0.15] }

In [ ]:
train(GradientBoostingRegressor(random_state=random_state), X_train, y_train, X_test, y_test, param_grid=params_grid, run_name="GradientBoosting")

# Résulats

**mean_squared_error**

In [ ]:
image = Image.open("images/mean_squared_error.png")
display(image)

L'erreur quadratique moyenne (MSE) entre les valeurs prédites et les valeurs réelles met en évidence les modèles qui commettent des erreurs importantes, en les pénalisant fortement. À partir de ce graphique, nous pouvons conclure que le modèle avec le meilleur pouvoir prédictif est le Gradient Boosting.

De manière générale, il est également observé que les modèles basés sur des arbres surpassent les modèles linéaires dans le cadre de ce projet. De plus, il est intéressant de noter que les modèles linéaires entraînés sur des données standardisées présentent des performances inférieures à ceux entraînés sur les données brutes. 

**mean_absolute_error**

In [ ]:
image = Image.open("images/mean_absolute_error.png")
display(image)

La moyenne des erreurs absolues (MAE) entre les valeurs réelles et les prédictions se montre plus robuste face aux valeurs extrêmes. À partir de ce graphique, nous pouvons conclure que le modèle le plus résilient face aux valeurs extrêmes est le Gradient Boosting.

De manière générale, il est également observé que les modèles basés sur des arbres continuent de surpasser les modèles linéaires dans le cadre de ce projet. Par ailleurs, il est intéressant de noter que, contrairement aux résultats précédents, les modèles linéaires entraînés sur des données standardisées offrent des performances supérieures à ceux entraînés sur des données brutes en termes de robustesse face aux valeurs manquantes.

**r2**

In [ ]:
image = Image.open("images/r2.png")
display(image)

Le coefficient de détermination r2 reflète la proportion de variance des données réelles expliquée par les modèles prédictifs. À partir de ce graphique, nous constatons un comportement similaire à celui observé avec l'erreur quadratique moyenne (MSE). Le modèle Gradient Boosting se distingue une fois de plus par son meilleur pouvoir explicatif.

De manière générale, les modèles basés sur des arbres surpassent nettement les modèles linéaires dans ce projet. Par ailleurs, il est confirmé que les modèles linéaires entraînés sur des données brutes affichent de meilleures performances que ceux entraînés sur des données standardisées, en cohérence avec les résultats obtenus pour la MSE.

**Conclusion :** Notre modèle à mettre en production est le Gradient Boosting.